In [ ]:
import pandas as pd
import numpy as np
import os
import json
from transformers import PegasusTokenizer, PegasusForConditionalGeneration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
os.listdir("/content/drive/MyDrive")

['Colab Notebooks',
 'test',
 'AI_lecture_summarizer',
 'train',
 'Copy of lecture_dataset_train_v3.csv']

In [ ]:
older_path = "/content/drive/MyDrive/train"
output_path = "/content/drive/MyDrive/AI_lecture_summarizer/new_dataset_training.csv"

OVERLAP_WORDS_COUNT = 140

MAX_CHUNK_WORDS = 680

def split_into_chunks(sentences, max_words=1000, overlap_words=0):
    chunks, current_chunk = [], []
    current_len = 0
    last_chunk_overlap = []

    for sent in sentences:
        sent_len = len(sent.split())

        if current_len + sent_len > max_words and current_chunk:
            chunks.append(" ".join(current_chunk))

            prev_chunk_words = " ".join(current_chunk).split()
            last_chunk_overlap = prev_chunk_words[-overlap_words:] if overlap_words > 0 else []

            current_chunk = [" ".join(last_chunk_overlap), sent] if last_chunk_overlap else [sent]

            current_len = len(last_chunk_overlap) + sent_len

        else:
            current_chunk.append(sent)
            current_len += sent_len

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

data_rows = []

for filename in os.listdir(folder_path):
    if filename.endswith(".json"):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            content = json.load(f)

        lecture_id = content.get("id", filename)
        title = content.get("title", "")

        segments = content.get("segmentation", [])
        all_sentences = [sent.strip() for seg in segments for sent in seg if sent.strip()]

        summarization = content.get("summarization", {})
        important_sentences = []
        for clip in summarization.values():
            if "summarization_data" in clip:
                for item in clip["summarization_data"]:
                    if item.get("label", 0) == 1:
                        important_sentences.append(item["sent"].strip())

        chunks = split_into_chunks(all_sentences, max_words=MAX_CHUNK_WORDS, overlap_words=OVERLAP_WORDS_COUNT)

        for i, chunk in enumerate(chunks):
            label_sents = [s for s in important_sentences if s in chunk]
            label = " ".join(label_sents) if label_sents else "0"

            data_rows.append({
                "lecture_id": lecture_id,
                "title": title,
                "chunk_id": i + 1,
                "input": chunk,
                "label": label
            })

df = pd.DataFrame(data_rows)
df.to_csv(output_path, index=False, encoding="utf-8")

print(f"✅ Reformatted {len(df)} chunks from {len(os.listdir(folder_path))} lecture files.")
print(f"📁 Saved to: {output_path}")

df.head()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-large and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

In [ ]:
pd.set_option('display.max_colwidth', None)
print(df.loc[0,['input','label']])